In [36]:
import pandas as pd
import numpy as np

In [3]:
np.random.seed(42)

In [4]:
full_dataset_df = pd.read_parquet("../../data/churn-prediction-25-26/train.parquet")

In [44]:
# Idea compute daily, bi-daily and weekly increase & decrease of time listened
# Compute the following features:

# - change of usage daily: percentage & total

## FOCUS ############################################
# - change of usage last 7 days: percentage & total
# - change of usage last 14 days: percentage & total
# - chnage of usage last 21 days: percentage & total
# - change of usage last 28 days: percentage & total
#####################################################

# - mean daily increase / decrease -> volatility

df = full_dataset_df.copy()
df["date"] = pd.to_datetime(df["time"]).dt.date
df = df.sort_values(["userId", "time"])
df_1 = df.groupby(["userId", "date"])["length"].sum().reset_index()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17499636 entries, 403570 to 15905873
Data columns (total 20 columns):
 #   Column         Dtype         
---  ------         -----         
 0   status         int64         
 1   gender         object        
 2   firstName      object        
 3   level          object        
 4   lastName       object        
 5   userId         object        
 6   ts             int64         
 7   auth           object        
 8   page           object        
 9   sessionId      int64         
 10  location       object        
 11  itemInSession  int64         
 12  userAgent      object        
 13  method         object        
 14  length         float64       
 15  song           object        
 16  artist         object        
 17  time           datetime64[us]
 18  registration   datetime64[us]
 19  date           object        
dtypes: datetime64[us](2), float64(1), int64(4), object(13)
memory usage: 2.7+ GB


In [45]:
df_1

,userId,date,length
0,1000025,2018-10-02,24922.31447
1,1000025,2018-10-03,77571.72247
2,1000025,2018-10-04,49228.60436
3,1000025,2018-10-05,5720.83247
4,1000025,2018-10-06,888.24027
...,...,...,...
205671,1999905,2018-10-26,7742.44334
205672,1999905,2018-10-31,10097.35279
205673,1999905,2018-11-10,0.00000
205674,1999905,2018-11-15,1397.23483


In [46]:
prev_7 = df_1[["userId", "date", "length"]].copy()
prev_7["date"] = prev_7["date"] + pd.Timedelta(days=7)   # shift forward so it lines up with current date
prev_7 = prev_7.rename(columns={"length": "length_7d_ago"})

prev_14 = df_1[["userId", "date", "length"]].copy()
prev_14["date"] = prev_14["date"] + pd.Timedelta(days=14)   # shift forward so it lines up with current date
prev_14 = prev_14.rename(columns={"length": "length_14d_ago"})

prev_21 = df_1[["userId", "date", "length"]].copy()
prev_21["date"] = prev_21["date"] + pd.Timedelta(days=21)   # shift forward so it lines up with current date
prev_21 = prev_21.rename(columns={"length": "length_21d_ago"})

prev_28 = df_1[["userId", "date", "length"]].copy()
prev_28["date"] = prev_28["date"] + pd.Timedelta(days=28)   # shift forward so it lines up with current date
prev_28 = prev_28.rename(columns={"length": "length_28d_ago"})



df_1 = df_1.merge(prev_7, on=["userId", "date"], how="left")
df_1["total_difference_7_day"] = df_1["length"] - df_1["length_7d_ago"]

df_1 = df_1.merge(prev_14, on=["userId", "date"], how="left")
df_1["total_difference_14_day"] = df_1["length"] - df_1["length_14d_ago"]

df_1 = df_1.merge(prev_21, on=["userId", "date"], how="left")
df_1["total_difference_21_day"] = df_1["length"] - df_1["length_21d_ago"]

df_1 = df_1.merge(prev_28, on=["userId", "date"], how="left")
df_1["total_difference_28_day"] = df_1["length"] - df_1["length_28d_ago"]

In [57]:
df_1["total_difference_7_day_mean"] = df_1.groupby(["userId"])["total_difference_7_day"].transform("mean")
df_1["total_difference_14_day_mean"] = df_1.groupby(["userId"])["total_difference_14_day"].transform("mean")
df_1["total_difference_21_day_mean"] = df_1.groupby(["userId"])["total_difference_21_day"].transform("mean")
df_1["total_difference_28_day_mean"] = df_1.groupby(["userId"])["total_difference_28_day"].transform("mean")

In [58]:
df_1

,userId,date,length,length_7d_ago,total_difference_7_day,length_14d_ago,total_difference_14_day,length_21d_ago,total_difference_21_day,length_28d_ago,total_difference_28_day,total_difference_7_day_mean,total_difference_14_day_mean,total_difference_21_day_mean,total_difference_28_day_mean
0,1000025,2018-10-02,24922.31447,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5059.867934,-10391.159843,NaN,NaN
1,1000025,2018-10-03,77571.72247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5059.867934,-10391.159843,NaN,NaN
2,1000025,2018-10-04,49228.60436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5059.867934,-10391.159843,NaN,NaN
3,1000025,2018-10-05,5720.83247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5059.867934,-10391.159843,NaN,NaN
4,1000025,2018-10-06,888.24027,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5059.867934,-10391.159843,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205671,1999905,2018-10-26,7742.44334,NaN,NaN,3081.81570,4660.62764,4978.48700,2763.95634,NaN,NaN,628.516915,5795.711310,1043.18375,2476.11629
205672,1999905,2018-10-31,10097.35279,NaN,NaN,3166.55781,6930.79498,NaN,NaN,NaN,NaN,628.516915,5795.711310,1043.18375,2476.11629
205673,1999905,2018-11-10,0.00000,NaN,NaN,NaN,NaN,677.58884,-677.58884,NaN,NaN,628.516915,5795.711310,1043.18375,2476.11629
205674,1999905,2018-11-15,1397.23483,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,628.516915,5795.711310,1043.18375,2476.11629


In [ ]:
df_1_train = df_1[["userId",
                   "total_difference_7_day_mean",
                   "total_difference_14_day_mean",
                   "total_difference_21_day_mean",
                   "total_difference_28_day_mean"]]

In [61]:
df_1_train = df_1_train.drop_duplicates()

In [62]:
df_1_train

,userId,total_difference_7_day_mean,total_difference_14_day_mean,total_difference_21_day_mean,total_difference_28_day_mean
0,1000025,-5059.867934,-10391.159843,NaN,NaN
14,1000035,1897.325254,6132.223746,5529.745650,2915.651360
35,1000083,11455.372502,NaN,NaN,NaN
44,1000103,NaN,NaN,NaN,NaN
47,1000164,3609.476105,7131.180760,9389.970895,9864.394307
...,...,...,...,...,...
205592,1999781,-3994.697518,8117.964504,3849.076925,14396.130896
205625,1999847,NaN,-2104.264120,NaN,NaN
205630,1999848,1794.327554,7576.302409,-109.104388,6900.444660
205656,1999892,7423.150705,4857.211730,21733.967810,NaN
